In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings("ignore")

In [4]:
parquet_path = Path("/content/drive/MyDrive/data.parquet")
df = pd.read_parquet(parquet_path)

print(df.head())
print(df.columns.tolist())
print(df.shape)

                                               audio            title  \
0  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...             Food   
1  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TIT2\x...     Electric Ave   
2  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...       This World   
3  {'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...          Freeway   
4  {'bytes': b'ID3\x04\x00\x00\x00\x00\x05YTIT2\x...  Spiritual Level   

       artist  
0        AWOL  
1        AWOL  
2        AWOL  
3   Kurt Vile  
4  Nicky Cook  
['audio', 'title', 'artist']
(1230, 3)


In [13]:
import tempfile
def resolve_audio_input(audio_value, parquet_path=None):
    if isinstance(audio_value, dict):
        if "bytes" in audio_value:
            audio_bytes = audio_value["bytes"]
            original_path = audio_value.get("path", "")

            suffix = ".bin"
            if isinstance(original_path, str) and "." in Path(original_path).name:
                suffix = Path(original_path).suffix or ".bin"

            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
            tmp.write(audio_bytes)
            tmp.flush()
            tmp.close()

            return tmp.name

    raise ValueError(f"Unsupported audio format: {type(audio_value)}")

In [6]:
yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")

class_map_path = tf.keras.utils.get_file(
    "yamnet_class_map.csv",
    "https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv"
)

class_map = pd.read_csv(class_map_path)
yamnet_labels = class_map["display_name"].tolist()

print("YAMNet loaded")
print("Labels:", len(yamnet_labels))

14096/14096 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
YAMNet loaded
Labels: 521


In [7]:
def load_audio(audio_path, sr=16000, duration=45):
    y, sr = librosa.load(audio_path, sr=sr, mono=True, duration=duration)
    if len(y) == 0:
        raise ValueError("Empty audio")
    return y, sr

In [8]:
def extract_dsp_features(y, sr):
    tempo = float(librosa.feature.tempo(y=y, sr=sr)[0])
    rms = librosa.feature.rms(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)

    key_idx = int(np.argmax(chroma.mean(axis=1)))
    key_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    key = key_names[key_idx]

    if tempo < 80:
        tempo_bucket = "slow"
    elif tempo < 120:
        tempo_bucket = "medium"
    else:
        tempo_bucket = "fast"

    rms_mean = float(rms.mean())
    if rms_mean < 0.03:
        energy_level = "low"
    elif rms_mean < 0.08:
        energy_level = "medium"
    else:
        energy_level = "high"

    centroid_mean = float(centroid.mean())
    if centroid_mean < 1500:
        brightness = "dark"
    elif centroid_mean < 3000:
        brightness = "balanced"
    else:
        brightness = "bright"

    return {
        "tempo_bpm": round(tempo, 2),
        "tempo_bucket": tempo_bucket,
        "energy_level": energy_level,
        "brightness": brightness,
        "rms_mean": round(rms_mean, 6),
        "spectral_centroid_mean": round(centroid_mean, 2),
        "zero_crossing_rate_mean": round(float(zcr.mean()), 6),
        "estimated_key": key,
    }

In [9]:
def extract_yamnet_tags(y, top_k=10, threshold=0.03):
    scores, embeddings, spectrogram = yamnet_model(y)
    mean_scores = tf.reduce_mean(scores, axis=0).numpy()
    top_idx = np.argsort(mean_scores)[::-1][:top_k]

    tags = []
    for i in top_idx:
        label = yamnet_labels[i]
        score = float(mean_scores[i])
        if score >= threshold:
            tags.append({"label": label, "score": round(score, 4)})
    return tags

In [16]:
row = df.iloc[0]

audio_path = resolve_audio_input(row["audio"], parquet_path)
print("temp/local audio path:", audio_path)
print("artist:", row["artist"])
print("title:", row["title"])

y, sr = load_audio(audio_path)
tags = extract_yamnet_tags(y, top_k=10, threshold=0.03)

tags

temp/local audio path: /tmp/tmpny1ocds7.mp3
artist: AWOL
title: Food


[{'label': 'Music', 'score': 0.8756},
 {'label': 'Rapping', 'score': 0.1052},
 {'label': 'Hip hop music', 'score': 0.0887},
 {'label': 'Singing', 'score': 0.0474}]

In [18]:
row = df.iloc[10]

audio_path = resolve_audio_input(row["audio"], parquet_path)
print("temp/local audio path:", audio_path)
print("artist:", row["artist"])
print("title:", row["title"])

y, sr = load_audio(audio_path)
tags = extract_yamnet_tags(y, top_k=10, threshold=0.03)

tags

temp/local audio path: /tmp/tmpdccnmxwk.mp3
artist: Abominog
title: Father's Day


[{'label': 'Music', 'score': 0.2881},
 {'label': 'Vehicle', 'score': 0.1958},
 {'label': 'Train', 'score': 0.1126},
 {'label': 'Rail transport', 'score': 0.1122},
 {'label': 'Train horn', 'score': 0.0977},
 {'label': 'Steam whistle', 'score': 0.088},
 {'label': 'Train whistle', 'score': 0.0813},
 {'label': 'Whistle', 'score': 0.0754},
 {'label': 'Railroad car, train wagon', 'score': 0.0754},
 {'label': 'Motor vehicle (road)', 'score': 0.0641}]

In [19]:
!pip -q install musicnn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 47.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Cannot install librosa==0.11.0, musicnn and musicnn==0.0.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
